<a href="https://colab.research.google.com/github/yedam823/AI_study/blob/main/EumSon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 셀 0: Drive mount
# ==============================================================================

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# 전처리 전용 설치 셀
!pip uninstall -y mediapipe protobuf tensorflow keras tf-keras
!pip install -q mediapipe opencv-python tqdm numpy==1.26.4 protobuf==4.25.8

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... canceled
ERROR: Operation cancelled by user


In [ ]:
import mediapipe as mp
import google.protobuf
import numpy as np

print("mediapipe:", mp.__version__)
print("protobuf:", google.protobuf.__version__)
print("numpy:", np.__version__)
print("has solutions:", hasattr(mp, "solutions"))

mp_holistic_module = mp.solutions.holistic

with mp_holistic_module.Holistic(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    refine_face_landmarks=False,
) as holistic:
    print("Holistic 생성 성공")

ModuleNotFoundError: No module named 'mediapipe'

In [ ]:
# ==============================================================================
# 셀 3: 경로 설정
# ==============================================================================

DATA_DIR = "/content/drive/MyDrive/2025_1116490-1130160_1368_교육시설"
OUTPUT_DIR = "/content/drive/MyDrive/landmarks"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(os.path.exists(DATA_DIR))
print(os.listdir(DATA_DIR)[:10])

True
['VXPAKOKS251120210.json', 'VXPAKOKS251120030.json', 'VXPAKOKS251120110.json', 'VXPAKOKS251120020.json', 'VXPAKOKS251120070.json', 'VXPAKOKS251120340.json', 'VXPAKOKS251120290.json', 'VXPAKOKS251120310.json', 'VXPAKOKS251120420.json', 'VXPAKOKS251120240.json']


In [ ]:
import os
import csv
import json
import glob
import logging
import traceback
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

import cv2
import numpy as np
import mediapipe as mp
from tqdm import tqdm

mp_holistic_module = mp.solutions.holistic

ModuleNotFoundError: No module named 'mediapipe'

In [ ]:
# ==============================================================================
# 셀 3: 상수 정의
# ==============================================================================

# 원본 데이터 경로 (필요 시 이 셀에서 직접 수정하세요)
DATA_DIR = (
    "/content/drive/MyDrive/"
    "2025_1116490-1130160_1368_교육시설"
)
OUTPUT_DIR = "/content/drive/MyDrive/landmarks"

# Pose: 33개 랜드마크 * (x, y, z, visibility) = 132
POSE_LANDMARK_COUNT = 33
POSE_FEATURE_DIM = 4  # x, y, z, visibility

# Hand: 21개 랜드마크 * (x, y, z) = 63 (좌/우 각각)
HAND_LANDMARK_COUNT = 21
HAND_FEATURE_DIM = 3  # x, y, z

# 최종 feature 차원 = (33*4) + (21*3) + (21*3) = 132 + 63 + 63 = 258
TOTAL_FEATURE_DIM = (
    POSE_LANDMARK_COUNT * POSE_FEATURE_DIM
    + HAND_LANDMARK_COUNT * HAND_FEATURE_DIM
    + HAND_LANDMARK_COUNT * HAND_FEATURE_DIM
)
assert TOTAL_FEATURE_DIM == 258, "feature 차원 계산이 258과 일치해야 합니다."

# JSON 내부 구조 키 이름
JSON_ROOT_KEY = "sign_script"
JSON_GESTURE_KEY = "sign_gestures_strong"

# 최소 프레임 길이 (이보다 짧은 gloss 구간은 노이즈로 간주하고 스킵)
MIN_VALID_FRAMES = 1

# 기본 FPS (영상에서 FPS를 읽어오지 못했을 때 사용하는 fallback 값)
FALLBACK_FPS = 30.0

In [ ]:
# ==============================================================================
# 셀 4: 로거 설정
# ==============================================================================

def setup_logger(output_dir: str) -> logging.Logger:
    """
    콘솔과 파일에 동시에 로그를 남기는 로거를 설정한다.

    Args:
        output_dir: 로그 파일(preprocess.log)이 저장될 디렉토리.

    Returns:
        설정이 완료된 Logger 인스턴스.
    """
    os.makedirs(output_dir, exist_ok=True)
    log_path = os.path.join(output_dir, "preprocess.log")

    logger = logging.getLogger("ksl_preprocess")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()  # Colab 재실행 시 핸들러 중복 방지

    formatter = logging.Formatter(
        fmt="[%(asctime)s] [%(levelname)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    return logger

In [ ]:
# ==============================================================================
# 셀 5: 데이터 클래스
# ==============================================================================

@dataclass
class GestureSegment:
    """JSON의 sign_gestures_strong 배열 원소 하나에 대응하는 구간 정보."""
    gloss_id: str
    start_sec: float
    end_sec: float
    start_frame: int = -1
    end_frame: int = -1
    landmarks: List[np.ndarray] = field(default_factory=list)


@dataclass
class ProcessResult:
    """영상 1개를 처리한 결과 요약."""
    video_path: str
    success: bool
    saved_samples: int = 0
    error_message: str = ""

In [ ]:
# ==============================================================================
# 셀 6: MediaPipe 랜드마크 추출 (Pose + Left Hand + Right Hand, Face 제외)
# ==============================================================================

def extract_landmarks(results) -> np.ndarray:
    """
    MediaPipe Holistic의 단일 프레임 처리 결과에서 Pose + Left Hand + Right Hand
    랜드마크만 추출하여 258차원 벡터로 변환한다. Face는 사용하지 않는다.

    검출되지 않은 부위는 0으로 채워 항상 고정된 258차원 shape을 보장한다.
    (LSTM 입력의 시퀀스 차원 일관성을 위해 반드시 필요한 처리)

    Args:
        results: holistic.process(image) 의 반환값.

    Returns:
        shape (258,) 의 float32 numpy 배열.
    """
    # ---- Pose (33 * 4 = 132) ----
    if results.pose_landmarks:
        pose = np.array(
            [[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark],
            dtype=np.float32,
        ).flatten()
    else:
        pose = np.zeros(POSE_LANDMARK_COUNT * POSE_FEATURE_DIM, dtype=np.float32)

    # ---- Left Hand (21 * 3 = 63) ----
    if results.left_hand_landmarks:
        left_hand = np.array(
            [[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark],
            dtype=np.float32,
        ).flatten()
    else:
        left_hand = np.zeros(HAND_LANDMARK_COUNT * HAND_FEATURE_DIM, dtype=np.float32)

    # ---- Right Hand (21 * 3 = 63) ----
    if results.right_hand_landmarks:
        right_hand = np.array(
            [[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark],
            dtype=np.float32,
        ).flatten()
    else:
        right_hand = np.zeros(HAND_LANDMARK_COUNT * HAND_FEATURE_DIM, dtype=np.float32)

    feature = np.concatenate([pose, left_hand, right_hand], axis=0)
    assert feature.shape[0] == TOTAL_FEATURE_DIM, (
        f"feature shape 오류: {feature.shape[0]} != {TOTAL_FEATURE_DIM}"
    )
    return feature

In [ ]:
# ==============================================================================
# 셀 7: 시간 <-> 프레임 변환
# ==============================================================================

def time_to_frame(time_sec: float, fps: float) -> int:
    """초 단위 시간을 프레임 번호로 변환한다 (반올림)."""
    return int(round(time_sec * fps))


def get_video_fps(cap: cv2.VideoCapture) -> float:
    """
    영상의 FPS를 자동 계산한다. cv2가 FPS를 0 또는 비정상 값으로 반환하는
    경우가 실제로 존재하므로, 이 경우 FALLBACK_FPS로 대체한다.
    """
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 1e-3 or fps > 240:
        logging.getLogger("ksl_preprocess").warning(
            "FPS 값이 비정상적입니다(%.3f). fallback FPS(%.1f)를 사용합니다.",
            fps if fps else -1.0, FALLBACK_FPS,
        )
        return FALLBACK_FPS
    return fps

In [ ]:
# ==============================================================================
# 셀 8: JSON 파싱
# ==============================================================================

def parse_gesture_segments(json_path: str) -> List[GestureSegment]:
    """
    JSON 파일에서 sign_script.sign_gestures_strong 배열을 읽어
    GestureSegment 리스트로 변환한다.

    Args:
        json_path: 대상 JSON 파일 경로.

    Returns:
        GestureSegment 리스트.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    root = data.get(JSON_ROOT_KEY)
    if root is None:
        raise KeyError(f"'{JSON_ROOT_KEY}' 키가 JSON에 존재하지 않습니다: {json_path}")

    gestures_raw = root.get(JSON_GESTURE_KEY)
    if gestures_raw is None:
        raise KeyError(f"'{JSON_GESTURE_KEY}' 키가 JSON에 존재하지 않습니다: {json_path}")

    segments: List[GestureSegment] = []
    for item in gestures_raw:
        start = float(item["start"])
        end = float(item["end"])
        gloss_id = str(item["gloss_id"])

        if end <= start:
            # 시간 역전 또는 0 길이 구간은 스킵 (데이터 오류 방어)
            continue

        segments.append(GestureSegment(gloss_id=gloss_id, start_sec=start, end_sec=end))

    return segments

In [ ]:
# ==============================================================================
# 셀 9: Manifest (재실행 시 스킵 판단용 메타데이터)
# ==============================================================================

class Manifest:
    """
    landmarks/manifest.csv 를 관리하는 클래스.

    - 이미 처리된 영상인지 판단 (해당 영상의 모든 gloss 구간이 이미 저장되었는지)
    - 다음에 사용할 전역 샘플 인덱스(video_N) 계산
    - 새로운 샘플을 기록
    """

    FIELDNAMES = [
        "sample_index", "video_file", "gesture_index",
        "gloss_id", "start_sec", "end_sec", "npy_path", "num_frames",
    ]

    def __init__(self, output_dir: str):
        self.path = os.path.join(output_dir, "manifest.csv")
        self._rows: List[Dict] = []
        self._video_gesture_count: Dict[str, int] = {}  # video_file -> 저장된 gesture 개수
        self._next_index = 0
        self._load()

    def _load(self) -> None:
        """기존 manifest.csv가 있으면 읽어서 내부 상태를 복원한다."""
        if not os.path.exists(self.path):
            return

        with open(self.path, "r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                self._rows.append(row)
                video_file = row["video_file"]
                self._video_gesture_count[video_file] = (
                    self._video_gesture_count.get(video_file, 0) + 1
                )
                self._next_index = max(self._next_index, int(row["sample_index"]) + 1)

    def is_video_fully_processed(self, video_file: str, expected_gesture_count: int) -> bool:
        """
        해당 영상의 모든 gloss 구간이 이미 manifest에 기록되어 있으면 True.
        """
        already = self._video_gesture_count.get(video_file, 0)
        return already >= expected_gesture_count and expected_gesture_count > 0

    def next_sample_index(self) -> int:
        """다음에 사용할 전역 샘플 인덱스를 반환하고 내부 카운터를 증가시킨다."""
        idx = self._next_index
        self._next_index += 1
        return idx

    def append(self, row: Dict) -> None:
        """새 샘플 1건을 메모리 버퍼와 카운터에 반영한다."""
        self._rows.append(row)
        video_file = row["video_file"]
        self._video_gesture_count[video_file] = (
            self._video_gesture_count.get(video_file, 0) + 1
        )

    def flush(self) -> None:
        """현재까지의 전체 내용을 manifest.csv에 다시 기록한다."""
        with open(self.path, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.FIELDNAMES)
            writer.writeheader()
            for row in self._rows:
                writer.writerow(row)

In [ ]:
# ==============================================================================
# 셀 10: 핵심 전처리 클래스
# ==============================================================================

class HolisticPreprocessor:
    """
    MediaPipe Holistic 객체를 1회만 생성하여 재사용하는 전처리기.

    영상 1개를 입력받아, 그 안에 포함된 모든 gloss 구간에 대해
    프레임을 순차 순회하면서 랜드마크를 추출하고, 각 구간을 하나의
    (T, 258) 샘플로 저장한다.
    """

    def __init__(self, output_dir: str, logger: logging.Logger,
                 min_detection_confidence: float = 0.5,
                 min_tracking_confidence: float = 0.5):
        self.output_dir = output_dir
        self.logger = logger

        # MediaPipe Holistic 객체는 파이프라인 전체에서 단 한 번만 생성한다.
        self.mp_holistic = mp_holistic_module
        self.holistic = self.mp_holistic.Holistic(
            static_image_mode=False,
            model_complexity=1,
            smooth_landmarks=True,
            min_detection_confidence=min_detection_confidence,
            min_tracking_confidence=min_tracking_confidence,
            refine_face_landmarks=False,
        )

    def close(self) -> None:
        """MediaPipe 리소스를 해제한다. 파이프라인 종료 시 반드시 호출."""
        self.holistic.close()

    def _process_frame(self, frame_bgr: np.ndarray) -> np.ndarray:
        """
        BGR 프레임 1장을 MediaPipe Holistic에 통과시켜 258차원 랜드마크
        벡터를 반환한다.
        """
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frame_rgb.flags.writeable = False  # 성능 최적화 (복사 방지)
        results = self.holistic.process(frame_rgb)
        return extract_landmarks(results)

    def process_video(
        self,
        video_path: str,
        json_path: str,
        manifest: Manifest,
    ) -> ProcessResult:
        """
        영상 1개 + JSON 1개를 처리하여 gloss 구간별 샘플을 저장한다.

        처리 흐름:
          1. JSON에서 gloss 구간 목록 파싱
          2. 이미 전부 처리된 영상이면 스킵
          3. 영상을 열고 FPS 계산, 각 구간의 start/end를 프레임 번호로 변환
          4. 프레임을 0번부터 순차적으로 grab() 하면서, 필요한 프레임만 decode
          5. 프레임이 속한 모든 활성 구간의 버퍼에 랜드마크 append
          6. 순회 종료 후 각 구간을 npy/txt로 저장하고 manifest에 기록
        """
        video_file = os.path.basename(video_path)

        # ---- 1. JSON 파싱 ----
        segments = parse_gesture_segments(json_path)
        if len(segments) == 0:
            return ProcessResult(
                video_path=video_path, success=False,
                error_message="유효한 gloss 구간이 JSON에 없음",
            )

        # ---- 2. 이미 처리된 영상이면 스킵 ----
        if manifest.is_video_fully_processed(video_file, len(segments)):
            self.logger.info("이미 처리 완료됨, 스킵: %s", video_file)
            return ProcessResult(video_path=video_path, success=True, saved_samples=0)

        # ---- 3. 영상 열기 및 FPS / 프레임 번호 계산 ----
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return ProcessResult(
                video_path=video_path, success=False,
                error_message="cv2.VideoCapture로 영상을 열 수 없음",
            )

        try:
            fps = get_video_fps(cap)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

            for seg in segments:
                seg.start_frame = max(0, time_to_frame(seg.start_sec, fps))
                seg.end_frame = time_to_frame(seg.end_sec, fps)
                if total_frames > 0:
                    seg.end_frame = min(seg.end_frame, total_frames - 1)

            max_frame_needed = max(seg.end_frame for seg in segments)

            # ---- 4~5. 프레임 순차 순회 (필요 없는 프레임은 decode 없이 skip) ----
            frame_idx = 0
            while frame_idx <= max_frame_needed:
                grabbed = cap.grab()
                if not grabbed:
                    self.logger.warning(
                        "영상이 예상보다 일찍 종료됨(frame=%d, video=%s)",
                        frame_idx, video_file,
                    )
                    break

                active_segments = [
                    seg for seg in segments
                    if seg.start_frame <= frame_idx <= seg.end_frame
                ]

                if active_segments:
                    ok, frame_bgr = cap.retrieve()
                    if ok:
                        landmark_vec = self._process_frame(frame_bgr)
                        for seg in active_segments:
                            seg.landmarks.append(landmark_vec)
                    else:
                        self.logger.warning(
                            "frame retrieve 실패 (frame=%d, video=%s)",
                            frame_idx, video_file,
                        )

                frame_idx += 1
        finally:
            # 메모리 누수 방지: 반드시 영상 캡처 리소스를 해제한다.
            cap.release()

        # ---- 6. 구간별 저장 ----
        saved_count = 0
        for gesture_index, seg in enumerate(segments):
            if len(seg.landmarks) < MIN_VALID_FRAMES:
                self.logger.warning(
                    "프레임 수 부족으로 스킵: video=%s gloss=%s (frames=%d)",
                    video_file, seg.gloss_id, len(seg.landmarks),
                )
                continue

            sample_index = manifest.next_sample_index()
            npy_name = f"video_{sample_index}.npy"
            txt_name = f"video_{sample_index}.txt"
            npy_path = os.path.join(self.output_dir, npy_name)
            txt_path = os.path.join(self.output_dir, txt_name)

            sequence = np.stack(seg.landmarks, axis=0).astype(np.float32)  # (T, 258)
            np.save(npy_path, sequence)
            with open(txt_path, "w", encoding="utf-8") as f:
                f.write(seg.gloss_id)

            manifest.append({
                "sample_index": sample_index,
                "video_file": video_file,
                "gesture_index": gesture_index,
                "gloss_id": seg.gloss_id,
                "start_sec": seg.start_sec,
                "end_sec": seg.end_sec,
                "npy_path": npy_name,
                "num_frames": sequence.shape[0],
            })
            saved_count += 1

        return ProcessResult(video_path=video_path, success=True, saved_samples=saved_count)

In [ ]:
def collect_video_json_pairs(data_dir: str) -> List[Tuple[str, str]]:
    logger = logging.getLogger("ksl_preprocess")
    mp4_files = sorted(glob.glob(os.path.join(data_dir, "*.mp4")))

    pairs: List[Tuple[str, str]] = []

    for video_path in mp4_files:
        stem = os.path.splitext(os.path.basename(video_path))[0]

        if stem.endswith("_L") or stem.endswith("_R"):
            continue

        base_name = os.path.splitext(video_path)[0]
        json_path = base_name + ".json"

        if os.path.exists(json_path):
            pairs.append((video_path, json_path))
        else:
            logger.warning("대응하는 JSON을 찾을 수 없어 제외됨: %s", video_path)

    return pairs

In [ ]:
# ==============================================================================
# 셀 12: 메인 파이프라인
# ==============================================================================

def run_preprocessing(data_dir: str, output_dir: str) -> None:
    """
    전체 전처리 파이프라인 실행:
      1. mp4-json 쌍 수집
      2. Manifest 로드 (재실행 스킵 판단)
      3. MediaPipe Holistic 1회 생성
      4. tqdm 진행률로 영상 순회, 예외 발생 시 다음 영상으로 계속 진행
      5. 성공/실패 개수 출력, 실패 목록 저장
      6. 자원 정리
    """
    os.makedirs(output_dir, exist_ok=True)
    logger = setup_logger(output_dir)

    logger.info("=" * 70)
    logger.info("한국수어 랜드마크 전처리 시작")
    logger.info("데이터 디렉토리: %s", data_dir)
    logger.info("출력 디렉토리: %s", output_dir)
    logger.info("=" * 70)

    pairs = collect_video_json_pairs(data_dir)
    logger.info("총 %d개의 (mp4, json) 쌍을 발견했습니다.", len(pairs))

    if len(pairs) == 0:
        logger.error("처리할 파일이 없습니다. 경로를 확인하세요: %s", data_dir)
        return

    manifest = Manifest(output_dir)
    preprocessor = HolisticPreprocessor(output_dir=output_dir, logger=logger)

    success_video_count = 0
    failed_video_count = 0
    total_saved_samples = 0
    failed_records: List[Tuple[str, str]] = []

    try:
        progress_bar = tqdm(pairs, desc="영상 처리 중", unit="video")
        for video_path, json_path in progress_bar:
            video_name = os.path.basename(video_path)
            progress_bar.set_postfix_str(video_name)

            try:
                result = preprocessor.process_video(video_path, json_path, manifest)

                if result.success:
                    success_video_count += 1
                    total_saved_samples += result.saved_samples
                else:
                    failed_video_count += 1
                    failed_records.append((video_name, result.error_message))
                    logger.error("처리 실패: %s (사유: %s)", video_name, result.error_message)

            except Exception as exc:
                # 하나의 영상에서 예외가 발생해도 파이프라인 전체는 계속 진행한다.
                failed_video_count += 1
                error_msg = f"{type(exc).__name__}: {exc}"
                failed_records.append((video_name, error_msg))
                logger.error(
                    "예외 발생, 다음 영상으로 계속 진행: %s\n%s",
                    video_name, traceback.format_exc(),
                )
            finally:
                manifest.flush()

    finally:
        # 메모리 누수 방지: MediaPipe Holistic 리소스를 반드시 해제한다.
        preprocessor.close()

    # ---- 실패 목록 저장 ----
    failed_list_path = os.path.join(output_dir, "failed_videos.txt")
    if failed_records:
        with open(failed_list_path, "w", encoding="utf-8") as f:
            for video_name, reason in failed_records:
                f.write(f"{video_name}\t{reason}\n")

    # ---- 최종 요약 출력 ----
    logger.info("=" * 70)
    logger.info("전처리 완료")
    logger.info("처리 성공 영상 수 : %d", success_video_count)
    logger.info("처리 실패 영상 수 : %d", failed_video_count)
    logger.info("생성된 총 샘플 수 : %d", total_saved_samples)
    if failed_records:
        logger.info("실패 목록 저장 위치: %s", failed_list_path)
    logger.info("=" * 70)

In [ ]:
# ==============================================================================
# 셀 13: 실행
# ==============================================================================
run_preprocessing(data_dir=DATA_DIR, output_dir=OUTPUT_DIR)

[2026-07-26 12:57:51] [INFO] ======================================================================
INFO:ksl_preprocess:======================================================================
[2026-07-26 12:57:51] [INFO] 한국수어 랜드마크 전처리 시작
INFO:ksl_preprocess:한국수어 랜드마크 전처리 시작
[2026-07-26 12:57:51] [INFO] 데이터 디렉토리: /content/drive/MyDrive/2025_1116490-1130160_1368_교육시설
INFO:ksl_preprocess:데이터 디렉토리: /content/drive/MyDrive/2025_1116490-1130160_1368_교육시설
[2026-07-26 12:57:51] [INFO] 출력 디렉토리: /content/drive/MyDrive/landmarks
INFO:ksl_preprocess:출력 디렉토리: /content/drive/MyDrive/landmarks
[2026-07-26 12:57:51] [INFO] ======================================================================
INFO:ksl_preprocess:======================================================================
[2026-07-26 12:57:51] [INFO] 총 701개의 (mp4, json) 쌍을 발견했습니다.
INFO:ksl_preprocess:총 701개의 (mp4, json) 쌍을 발견했습니다.
영상 처리 중: 100%|██████████| 701/701 [1:39:12<00:00,  8.49s/video, VXPAKOKS251123490.mp4]
[2026-07-26 14:37:04] [INFO

In [ ]:
# ==============================================================================
# 셀 15: Import
# ==============================================================================
!pip install -q --upgrade tensorflow scikit-learn

import os
import csv
import pickle
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from collections import Counter

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-hub 0.16.1 requires tf-keras>=2.14.1, which is not installed.
dopamine-rl 4.1.2 requires tf-keras>=2.18.0, which is not installed.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.21.0 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.21.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.1 which is incompatible.


In [ ]:
# ==============================================================================
# 셀 16: 학습 설정
# ==============================================================================

LANDMARK_DIR = OUTPUT_DIR
MANIFEST_PATH = os.path.join(LANDMARK_DIR, "manifest.csv")

MODEL_SAVE_PATH = os.path.join(LANDMARK_DIR, "ksl_lstm_model.keras")
LABEL_ENCODER_PATH = os.path.join(LANDMARK_DIR, "label_encoder.pkl")

SEQ_LEN = 60
FEATURE_DIM = 258

POSE_DIM = 33 * 4          # 132
LEFT_HAND_START = POSE_DIM
LEFT_HAND_END = POSE_DIM + 21 * 3       # 195
RIGHT_HAND_START = LEFT_HAND_END
RIGHT_HAND_END = RIGHT_HAND_START + 21 * 3  # 258

MIN_SAMPLES_PER_CLASS = 10
BATCH_SIZE = 32
EPOCHS = 80

In [ ]:
# ==============================================================================
# 셀 17: 시퀀스 길이 고정 + 손 인식도 계산
# ==============================================================================

def resize_sequence(sequence, seq_len=SEQ_LEN):
    """
    각 단어 구간마다 프레임 수 T가 다르므로 LSTM 입력 길이를 고정한다.
    너무 길면 균등 샘플링, 너무 짧으면 0 padding.
    """
    sequence = np.asarray(sequence, dtype=np.float32)

    if len(sequence) == 0:
        return np.zeros((seq_len, FEATURE_DIM), dtype=np.float32)

    if len(sequence) >= seq_len:
        indices = np.linspace(0, len(sequence) - 1, seq_len)
        indices = np.round(indices).astype(int)
        return sequence[indices]

    padded = np.zeros((seq_len, FEATURE_DIM), dtype=np.float32)
    padded[:len(sequence)] = sequence
    return padded


def compute_hand_score(sequence):
    """
    손 인식도:
    전체 프레임 중 왼손 또는 오른손 좌표가 0이 아닌 프레임의 비율.

    반환값:
      0.0 = 손이 거의 인식되지 않음
      1.0 = 모든 프레임에서 손이 인식됨
    """
    sequence = np.asarray(sequence, dtype=np.float32)

    if len(sequence) == 0:
        return 0.0

    left_hand = sequence[:, LEFT_HAND_START:LEFT_HAND_END]
    right_hand = sequence[:, RIGHT_HAND_START:RIGHT_HAND_END]

    left_detected = np.any(np.abs(left_hand) > 1e-6, axis=1)
    right_detected = np.any(np.abs(right_hand) > 1e-6, axis=1)

    hand_detected = np.logical_or(left_detected, right_detected)

    return float(np.mean(hand_detected))

In [ ]:
import os, glob

DATA_DIR = "/content/drive/MyDrive/2025_1116490-1130160_1368_교육시설"
OUTPUT_DIR = "/content/drive/MyDrive/landmarks"

print("DATA_DIR exists:", os.path.exists(DATA_DIR))
print("mp4 count:", len(glob.glob(DATA_DIR + "/*.mp4")))
print("json count:", len(glob.glob(DATA_DIR + "/*.json")))

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR)

DATA_DIR exists: True
mp4 count: 2101
json count: 1368
OUTPUT_DIR: /content/drive/MyDrive/landmarks


In [ ]:
# ==============================================================================
# 셀 18: manifest.csv에서 학습 데이터 로드
# ==============================================================================

def load_manifest_dataset(manifest_path, landmark_dir):
    X = []
    y_word = []
    y_hand_score = []

    with open(manifest_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            npy_path = row["npy_path"]
            gloss_id = row["gloss_id"]

            full_npy_path = os.path.join(landmark_dir, npy_path)

            if not os.path.exists(full_npy_path):
                continue

            sequence = np.load(full_npy_path).astype(np.float32)

            hand_score = compute_hand_score(sequence)
            sequence = resize_sequence(sequence, SEQ_LEN)

            X.append(sequence)
            y_word.append(gloss_id)
            y_hand_score.append(hand_score)

    X = np.asarray(X, dtype=np.float32)
    y_hand_score = np.asarray(y_hand_score, dtype=np.float32)

    return X, y_word, y_hand_score


X, y_word, y_hand_score = load_manifest_dataset(MANIFEST_PATH, LANDMARK_DIR)

print("X shape:", X.shape)
print("단어 라벨 수:", len(y_word))
print("손 인식도 shape:", y_hand_score.shape)
print("손 인식도 예시:", y_hand_score[:10])

In [ ]:
# ==============================================================================
# 셀 19: 샘플이 너무 적은 단어 제거 + 라벨 인코딩
# ==============================================================================

label_counter = Counter(y_word)

valid_labels = {
    label for label, count in label_counter.items()
    if count >= MIN_SAMPLES_PER_CLASS
}

keep_indices = [
    i for i, label in enumerate(y_word)
    if label in valid_labels
]

X = X[keep_indices]
y_word = [y_word[i] for i in keep_indices]
y_hand_score = y_hand_score[keep_indices]

label_encoder = LabelEncoder()
y_class = label_encoder.fit_transform(y_word)

num_classes = len(label_encoder.classes_)

with open(LABEL_ENCODER_PATH, "wb") as f:
    pickle.dump(label_encoder, f)

print("최종 샘플 수:", len(X))
print("최종 단어 클래스 수:", num_classes)
print("단어 예시:", label_encoder.classes_[:20])

In [ ]:
# ==============================================================================
# 셀 20: Train / Validation 분리
# ==============================================================================

X_train, X_val, y_class_train, y_class_val, y_hand_train, y_hand_val = train_test_split(
    X,
    y_class,
    y_hand_score,
    test_size=0.2,
    random_state=42,
    stratify=y_class
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

NameError: name 'train_test_split' is not defined

In [ ]:
# ==============================================================================
# 셀 21: LSTM 멀티 출력 모델
# ==============================================================================

def build_lstm_model(seq_len, feature_dim, num_classes):
    inputs = tf.keras.layers.Input(
        shape=(seq_len, feature_dim),
        name="landmark_input"
    )

    x = tf.keras.layers.Masking(mask_value=0.0)(inputs)

    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(128, return_sequences=True)
    )(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(64)
    )(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    word_output = tf.keras.layers.Dense(
        num_classes,
        activation="softmax",
        name="word_output"
    )(x)

    hand_score_output = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        name="hand_score_output"
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=[word_output, hand_score_output]
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss={
            "word_output": "sparse_categorical_crossentropy",
            "hand_score_output": "mse",
        },
        loss_weights={
            "word_output": 1.0,
            "hand_score_output": 0.2,
        },
        metrics={
            "word_output": ["accuracy"],
            "hand_score_output": ["mae"],
        }
    )

    return model


model = build_lstm_model(
    seq_len=SEQ_LEN,
    feature_dim=FEATURE_DIM,
    num_classes=num_classes
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ landmark_input      │ (None, 60, 258)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 60, 258)   │          0 │ landmark_input[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_1 (Masking) │ (None, 60, 258)   │          0 │ landmark_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_3 (Any)         │ (None, 60)        │          0 │ not_equal_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 60, 256)   │    396,288 │ masking_1[0][0],  │
│ (Bidirectional)     │                   │            │ any_3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 60, 256)   │          0 │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 128)       │    164,352 │ dropout_3[0][0],  │
│ (Bidirectional)     │                   │            │ any_3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 128)       │          0 │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     16,512 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_output (Dense) │ (None, 125)       │     16,125 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hand_score_output   │ (None, 1)         │        129 │ dropout_5[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 593,918 (2.27 MB)

 Trainable params: 593,662 (2.26 MB)

 Non-trainable params: 256 (1.00 KB)

In [ ]:
# ==============================================================================
# 셀 22: 학습
# ==============================================================================

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor="val_word_output_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_word_output_accuracy",
        mode="max",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-5,
        verbose=1
    )
]

history = model.fit(
    X_train,
    {
        "word_output": y_class_train,
        "hand_score_output": y_hand_train,
    },
    validation_data=(
        X_val,
        {
            "word_output": y_class_val,
            "hand_score_output": y_hand_val,
        }
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)

Epoch 1/80
115/116 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - hand_score_output_loss: 0.2707 - hand_score_output_mae: 0.4692 - loss: 5.1830 - word_output_accuracy: 0.0139 - word_output_loss: 5.1288
Epoch 1: val_word_output_accuracy improved from None to 0.19848, saving model to /content/drive/MyDrive/landmarks/ksl_lstm_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/landmarks/ksl_lstm_model.keras
116/116 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - hand_score_output_loss: 0.2383 - hand_score_output_mae: 0.4421 - loss: 5.0072 - word_output_accuracy: 0.0293 - word_output_loss: 4.9556 - val_hand_score_output_loss: 0.1187 - val_hand_score_output_mae: 0.3438 - val_loss: 4.4525 - val_word_output_accuracy: 0.1985 - val_word_output_loss: 4.4287 - learning_rate: 0.0010
Epoch 2/80
114/116 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - hand_score_output_loss: 0.1148 - hand_score_output_mae: 0.2949 - loss: 4.3678 - word_output_accuracy: 0.1333 - word_output_loss: 4.3449
Epoch 2: val_word_output_accura

In [ ]:
# ==============================================================================
# 셀 24: 예측 함수
# ==============================================================================

def predict_landmark_sequence(sequence):
    """
    sequence:
      원본 (T, 258) 또는 이미 resize된 (60, 258) 좌표 배열
    """
    if sequence.shape[0] != SEQ_LEN:
        sequence = resize_sequence(sequence, SEQ_LEN)

    sequence = sequence.astype(np.float32)

    pred_word, pred_hand_score = model.predict(sequence[None, ...], verbose=0)

    class_index = int(np.argmax(pred_word[0]))
    word = label_encoder.inverse_transform([class_index])[0]

    word_confidence = float(pred_word[0][class_index])
    hand_score = float(pred_hand_score[0][0])

    return {
        "word": word,
        "word_confidence": word_confidence,
        "hand_score": hand_score,
    }


sample_idx = 0
result = predict_landmark_sequence(X_val[sample_idx])

print("예측 결과:", result)
print("정답 단어:", label_encoder.inverse_transform([y_class_val[sample_idx]])[0])
print("정답 손 인식도:", float(y_hand_val[sample_idx]))

NameError: name 'X_val' is not defined

In [ ]:
# ==============================================================================
# 모델 검증
# ==============================================================================

import os
import pickle
import numpy as np
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    top_k_accuracy_score
)

# ------------------------------------------------------------------------------
# 1. 저장된 최적 모델 불러오기
# ------------------------------------------------------------------------------

MODEL_SAVE_PATH = os.path.join(
    LANDMARK_DIR,
    "ksl_lstm_model.keras"
)

LABEL_ENCODER_PATH = os.path.join(
    LANDMARK_DIR,
    "label_encoder.pkl"
)

print("모델 경로:", MODEL_SAVE_PATH)
print("라벨 인코더 경로:", LABEL_ENCODER_PATH)

model = tf.keras.models.load_model(MODEL_SAVE_PATH)

with open(LABEL_ENCODER_PATH, "rb") as f:
    label_encoder = pickle.load(f)

print("모델 로드 완료")
print("클래스 수:", len(label_encoder.classes_))


# ------------------------------------------------------------------------------
# 2. Validation 데이터 전체 평가
# ------------------------------------------------------------------------------

results = model.evaluate(
    X_val,
    {
        "word_output": y_class_val,
        "hand_score_output": y_hand_val,
    },
    batch_size=BATCH_SIZE,
    verbose=1
)

print("\n==============================")
print("Validation 평가 결과")
print("==============================")

for name, value in zip(model.metrics_names, results):
    print(f"{name}: {value:.4f}")


# ------------------------------------------------------------------------------
# 3. 단어 분류 예측
# ------------------------------------------------------------------------------

pred_word, pred_hand_score = model.predict(
    X_val,
    batch_size=BATCH_SIZE,
    verbose=1
)

# 가장 높은 확률의 클래스
y_pred = np.argmax(pred_word, axis=1)

# 실제 정답
y_true = np.asarray(y_class_val)


# ------------------------------------------------------------------------------
# 4. Accuracy
# ------------------------------------------------------------------------------

accuracy = accuracy_score(y_true, y_pred)

print("\n==============================")
print("단어 인식 성능")
print("==============================")
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")


# ------------------------------------------------------------------------------
# 5. Top-5 Accuracy
# ------------------------------------------------------------------------------

top5_accuracy = top_k_accuracy_score(
    y_true,
    pred_word,
    k=min(5, pred_word.shape[1]),
    labels=np.arange(pred_word.shape[1])
)

print(f"Top-5 Accuracy: {top5_accuracy:.4f}")
print(f"Top-5 Accuracy: {top5_accuracy * 100:.2f}%")


# ------------------------------------------------------------------------------
# 6. 손 인식도 MAE
# ------------------------------------------------------------------------------

hand_pred = pred_hand_score[:, 0]

hand_mae = np.mean(
    np.abs(hand_pred - y_hand_val)
)

print("\n==============================")
print("손 인식도 성능")
print("==============================")
print(f"MAE: {hand_mae:.4f}")


# ------------------------------------------------------------------------------
# 7. Classification Report
# ------------------------------------------------------------------------------

print("\n==============================")
print("Classification Report")
print("==============================")

# 실제 validation 데이터에 등장한 클래스만 사용
unique_classes = np.unique(
    np.concatenate([y_true, y_pred])
)

target_names = label_encoder.inverse_transform(
    unique_classes
)

print(
    classification_report(
        y_true,
        y_pred,
        labels=unique_classes,
        target_names=target_names,
        zero_division=0,
        digits=4
    )
)


# ------------------------------------------------------------------------------
# 8. 샘플별 예측 확인
# ------------------------------------------------------------------------------

print("\n==============================")
print("샘플 예측 결과")
print("==============================")

num_samples = min(20, len(X_val))

for i in range(num_samples):

    predicted_class = y_pred[i]

    predicted_word = label_encoder.inverse_transform(
        [predicted_class]
    )[0]

    true_word = label_encoder.inverse_transform(
        [y_true[i]]
    )[0]

    confidence = float(pred_word[i][predicted_class])

    predicted_hand = float(hand_pred[i])
    true_hand = float(y_hand_val[i])

    correct = predicted_word == true_word

    print(
        f"[{i:03d}] "
        f"정답={true_word} | "
        f"예측={predicted_word} | "
        f"확률={confidence:.4f} | "
        f"손인식도={predicted_hand:.4f} "
        f"(정답={true_hand:.4f}) | "
        f"{'O' if correct else 'X'}"
    )


# ------------------------------------------------------------------------------
# 9. 오인식 샘플만 출력
# ------------------------------------------------------------------------------

print("\n==============================")
print("오인식 샘플")
print("==============================")

wrong_indices = np.where(y_true != y_pred)[0]

print(f"전체 검증 샘플: {len(y_true)}")
print(f"오인식 샘플: {len(wrong_indices)}")
print(
    f"오인식률: "
    f"{len(wrong_indices) / len(y_true) * 100:.2f}%"
)

for i in wrong_indices[:30]:

    predicted_class = y_pred[i]

    predicted_word = label_encoder.inverse_transform(
        [predicted_class]
    )[0]

    true_word = label_encoder.inverse_transform(
        [y_true[i]]
    )[0]

    confidence = float(pred_word[i][predicted_class])

    print(
        f"샘플 {i}: "
        f"정답={true_word}, "
        f"예측={predicted_word}, "
        f"확률={confidence:.4f}"
    )


NameError: name 'LANDMARK_DIR' is not defined